In [5]:
import logging
logging.basicConfig(level=logging.WARNING)
root_logger = logging.getLogger()
root_logger.setLevel(logging.WARNING)
for name in ("transformers", "tokenizers", "torch"):
    logging.getLogger(name).setLevel(logging.WARNING)

import torch
import torch.nn as nn
import json
import os
import pandas as pd
from models.lstm_proj_diff import LSTM as LSTM2
from models.transformer import Transformer
from utils.dataset import TranslateDataset
from torch.utils.data import DataLoader
import tokenizers
import numpy as np
from tqdm import tqdm

In [6]:
np.exp(1.05)

np.float64(2.857651118063164)

In [7]:
tokenizer = tokenizers.Tokenizer.from_file("artifacts/tokenizer_50000.json")

In [8]:
PATH_MODELS = "artifacts"

models = {}
for file in os.listdir(PATH_MODELS):
    if file.endswith(".pt"):
        model_name = file.split(".")[0].replace('model_', '')
        # keep only models with version >= 47
        try:
            ver = int(model_name.lstrip('v')) if model_name.startswith('v') else None
        except ValueError:
            ver = None
        if ver is None or ver not in (50, 56, 62, 64, 65):
            continue
        print(f'Lendo {model_name}')
        model_path = os.path.join(PATH_MODELS, file)
        model_config_path = os.path.join("configs", f"{model_name}_config.json")
        
        with open(model_config_path, 'r', encoding='utf-8') as f:
            model_config = json.load(f)
        
        if 'architecture' in model_config and model_config['architecture'] == 'transformer':
            model = Transformer(
                embedding_dim=model_config["embedding_dim"],
                encoder_num_layers=model_config["encoder_num_layers"],
                decoder_num_layers=model_config["decoder_num_layers"],
                encoder_hidden_dim=model_config["encoder_hidden_dim"],
                decoder_hidden_dim=model_config["decoder_hidden_dim"],
                encoder_num_heads=model_config["encoder_num_heads"],
                decoder_num_heads=model_config["decoder_num_heads"],
                encoder_dropout=model_config["encoder_dropout"],
                decoder_dropout=model_config["decoder_dropout"],
                vocab_size=model_config["vocab_size"] if "vocab_size" in model_config else 50000,
                kv_cache=True,
            )
        else:
            model = LSTM2(embedding_dim=model_config["embedding_dim"],
                        encoder_bidirectional=model_config["encoder_bidirectional"],
                        encoder_hidden_dim=model_config["encoder_hidden_dim"],
                        encoder_num_layers=model_config["encoder_num_layers"],
                        decoder_hidden_dim=model_config["decoder_hidden_dim"],
                        decoder_num_layers=model_config["decoder_num_layers"],
                        encoder_dropout=model_config["encoder_dropout"],
                        decoder_dropout=model_config["decoder_dropout"],
                        vocab_size=model_config["vocab_size"] if "vocab_size" in model_config else 10000, 
                        attention = model_config["attention"] if "attention" in model_config else False,
                        pad_idx=0)
        model.load_state_dict(torch.load(model_path))
        model.to("cuda" if torch.cuda.is_available() else "cpu")
        model.eval()
        models[model_name] = model

Lendo v50
Lendo v56
Lendo v62
Lendo v64
Lendo v65


In [9]:
frase = """The way I danced with you."""
for model_name, model in models.items():
    print(f"Modelo: {model_name}")
    
    device = next(model.parameters()).device
    model.eval()
    model.to(device)

    frase_tokenized = tokenizer.encode(frase).ids
    frase_tokenized = torch.tensor(frase_tokenized, dtype=torch.long).unsqueeze(0).to(device)
    
    id_tokens_translated = model_name, model.predict(
        frase_tokenized,
        bos_token_id=5,
        eos_token_id=6,
        max_len=300
    )

    frase_translated = tokenizer.decode(id_tokens_translated[1][0].tolist(), skip_special_tokens=False)
    print(f"Frase traduzida com o {model_name}: {frase_translated}")

Modelo: v50
Frase traduzida com o v50: A jeito que eu dançcom você . <EOS>
Modelo: v56
Frase traduzida com o v56: A maneira de dançar com você . <EOS>
Modelo: v62
Frase traduzida com o v62: <BOS>O que eu faço com você , como você . <EOS>
Modelo: v64
Frase traduzida com o v64: <BOS>A forma como dancei com você dançava . com você . <EOS>
Modelo: v65
Frase traduzida com o v65: <BOS>A maneira como dançava com você , dançavamos com você . <EOS>
